In [1]:
%run 0_1_load_paths.ipynb

In [2]:
import re

import commute_dm.core
import commute_dm.utils
import credentials
import momapy_kb.lpg.backends.neo4j
import momapy_kb.lpg.session
import pandas

In [3]:
backend = momapy_kb.lpg.backends.neo4j.Neo4jBackend(
    hostname=credentials.NEO4J_URI,
    username=credentials.NEO4J_USERNAME,
    password=credentials.NEO4J_PASSWORD,
    notifications_min_severity="off",
)
session = momapy_kb.lpg.session.Session(backend)

## Load the immuno targets list

In [4]:
IMMUNO_TARGETS_DIR = RESULTS_DIR / "immuno_targets/"
IMMUNO_TARGETS_FILE = DATA_DIR / "immuno_targets/immuno_targets.csv"
IMMUNO_TARGETS_GRAPHS_DIR = IMMUNO_TARGETS_DIR / "graphs/"
IMMUNO_TARGETS_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
targets_df = pandas.read_csv(IMMUNO_TARGETS_FILE, sep="\t")
targets_df = targets_df.dropna(subset=["UniProtID"]).reset_index(drop=True)
targets_df["UniProtIDs"] = targets_df["UniProtID"].apply(
    lambda raw: [
        uniprot_id.strip()
        for uniprot_id in re.split(r"[|;]", raw)
        if uniprot_id.strip()
    ]
)
all_uniprot_ids = sorted(
    {
        uniprot_id
        for uniprot_ids in targets_df["UniProtIDs"]
        for uniprot_id in uniprot_ids
    }
)

## Influence graphs around the targets matched in both maps

For every immuno target that has at least one matching model element in *both* the COVID and the PD maps, we build the influence graph upstream of the COVID seeds and downstream of the PD seeds (same procedure as for the COVID/PD interface, see `4_1_make_interface_graphs`).

In [6]:
MAX_LEVELS = [2, 3, 4, 5, 6]
MIN_N_NODES = 5

In [7]:
query = """
WITH $immuno_ids AS immuno_ids
MATCH
    (collection:Collection)-[:HAS_ENTRY]->(collection_entry:CollectionEntry),
    (collection_entry)-[:HAS_ELEMENT_TO_ANNOTATIONS]->(annotations:Mapping),
    (collection_entry)-[:HAS_OBJ]->(map:CellDesignerMap)-[:HAS_MODEL]->(model:CellDesignerModel),
    (annotations)-[:HAS_ITEM]->(annotations_item:Item),
    (annotations_item)-[:HAS_KEY]->(protein:Protein),
    (annotations_item)-[:HAS_VALUE]->(annotations_bag:Bag),
    (annotations_bag)-[:HAS_ITEM]->(rdf_annotation:RDFAnnotation)
UNWIND rdf_annotation.resources AS resource
WITH immuno_ids, collection, collection_entry, protein, resource
WHERE resource STARTS WITH 'urn:miriam:uniprot:'
  AND split(resource, ':')[-1] IN immuno_ids
WITH split(resource, ':')[-1] AS uniprot_id,
     collect(DISTINCT [collection, collection_entry, protein]) AS items
RETURN uniprot_id, items
"""
nodes_results = session.execute_query(query, params={"immuno_ids": all_uniprot_ids})
nodes_by_uniprot_id = {row["uniprot_id"]: row["items"] for row in nodes_results}

In [8]:
interface = {}
for _, row in targets_df.iterrows():
    target = row["Target"]
    nodes_with_context = []
    seen_protein_ids = set()
    collections_present = set()
    for uniprot_id in row["UniProtIDs"]:
        for (
            collection_node,
            collection_entry_node,
            protein_node,
        ) in nodes_by_uniprot_id.get(uniprot_id, []):
            if protein_node.element_id in seen_protein_ids:
                continue
            seen_protein_ids.add(protein_node.element_id)
            collections_present.add(collection_node["name"])
            nodes_with_context.append(
                {
                    "collection": collection_node,
                    "entry": collection_entry_node,
                    "node": protein_node,
                }
            )
    if {"COVID_DM_CD", "PD_DM_CD"}.issubset(collections_present):
        interface[target] = nodes_with_context
print(f"{len(interface)} targets matched in both COVID and PD maps")

12 targets matched in both COVID and PD maps


In [9]:
commute_dm.utils.remake_dir(IMMUNO_TARGETS_GRAPHS_DIR)
commute_dm.core.make_and_render_igs_from_interface(
    session=session,
    interface=interface,
    output_dir_path=IMMUNO_TARGETS_GRAPHS_DIR,
    max_levels=MAX_LEVELS,
    covid_nodes_color="lightblue",
    pd_nodes_color="lightgreen",
    common_nodes_color="goldenrod",
    interface_nodes_color="red",
    min_n_nodes=MIN_N_NODES,
)